# HONEST-RL-Calibrator — GRPO Training Notebook

**Goal:** Teach `Qwen2.5-3B-Instruct` to be *calibrated* — confident when right, uncertain when wrong.

> ⚙️ Recommended runtime: **A100 GPU** (17GB+ VRAM). T4 works but is slower.

---

## Step 1 — Install Dependencies

In [ ]:
# Install Unsloth first (pinned to a stable Colab-compatible version)
!pip install unsloth --quiet
!pip install -U trl peft transformers datasets bitsandbytes accelerate aiohttp python-constraint --quiet
print('✓ Dependencies installed')

## Step 2 — Clone Repository

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/HONEST-Env.git'  # ← update this

if not os.path.exists('/content/HONEST-Env'):
    !git clone {REPO_URL} /content/HONEST-Env
else:
    !git -C /content/HONEST-Env pull

%cd /content/HONEST-Env
print('✓ Repository ready')

## Step 3 — Configure Environment Variables

In [ ]:
import os

# ── Required: URL of your deployed HONEST-RL-Calibrator HuggingFace Space ──
# Leave empty to use local Brier score reward (no server needed)
os.environ['HONEST_ENV_URL'] = ''   # e.g. 'https://your-space.hf.space'

# ── Optional: W&B key for experiment tracking ───────────────────────────────
os.environ['WANDB_API_KEY'] = ''    # get from https://wandb.ai/authorize

# ── Optional: HuggingFace token (needed to push adapter to Hub later) ───────
os.environ['HF_TOKEN'] = ''        # get from https://huggingface.co/settings/tokens

print('✓ Environment variables set')

## Step 4 — Verify GPU

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 5 — Dry Run (Sanity Check)

This loads the model AND builds the full dataset but skips actual training.  
Run this first to confirm everything is wired correctly before the long training run.

In [ ]:
!python training/train_grpo.py --dry-run 2>&1 | tail -20

## Step 6 — Full GRPO Training

> ⏱️ Estimated time: ~60–90 min on A100, ~3–4 hrs on T4.

Reward distribution is logged every 10 steps. Watch `mean` reward climb from ~-0.5 toward 0.0 as calibration improves.

In [ ]:
# Remove --no-wandb if you set WANDB_API_KEY above
!python training/train_grpo.py --no-wandb 2>&1

## Step 7 — Post-training Evaluation

In [ ]:
# Point baseline_eval at the newly merged model
import subprocess, re

# Merge adapters into a standalone model first
merge_script = '''
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained(
    "./honest-qwen-3b-grpo/final_adapters",
    max_seq_length=2048, load_in_4bit=True
)
model.save_pretrained_merged("./honest-qwen-3b-merged", tok, save_method="merged_16bit")
print("Merged model saved to ./honest-qwen-3b-merged")
'''
exec(merge_script)

In [ ]:
# Run evaluation against the merged model
import re

# Patch MODEL_ID in baseline_eval.py to point at the merged folder
with open('eval/baseline_eval.py', 'r') as f:
    src = f.read()
src = re.sub(r'MODEL_ID\s*=\s*".*?"', 'MODEL_ID = "./honest-qwen-3b-merged"', src)
with open('eval/baseline_eval.py', 'w') as f:
    f.write(src)

!python eval/baseline_eval.py --output eval/after_rl_results.json --verbose 2>&1

## Step 8 — Plot Before / After Comparison

In [ ]:
!pip install matplotlib seaborn --quiet
from eval.plot_reliability import plot_comparison

out = plot_comparison(
    'eval/baseline_results.json',
    'eval/after_rl_results.json',
    label_before='Before RL (Baseline)',
    label_after='After GRPO Training',
)
print(f'Comparison plot saved to: {out}')

# Display inline in Colab
from IPython.display import Image
Image(out)

## Step 9 — (Optional) Push Adapter to HuggingFace Hub

In [ ]:
HF_USERNAME  = 'your-username'   # ← update
HF_REPO_NAME = 'honest-qwen-3b-grpo'

!huggingface-cli login --token $HF_TOKEN
!cd honest-qwen-3b-grpo/final_adapters && \
    huggingface-cli upload {HF_USERNAME}/{HF_REPO_NAME} . .